# 05b — Kinatrax 關節角匯入與座標慣例映射
### *Angle-driven import + convention mapping — 本章的核心*

你的 Kinatrax 輸出的是**已處理好的關節角度**，不是 raw marker 軌跡。所以：

- ❌ 我們**不跑 IK**（IK 是把 marker 變角度；你已經有角度了）。
- ⚠️ 真正的風險是 **座標慣例不一致 (convention mismatch)**：Kinatrax 的 segment 定義、
  joint axis、Euler/Cardan 順序、sign、zero pose、單位，幾乎一定 **不等於** OpenSim 模型的
  coordinate 定義。把角度直接倒進同名 coordinate → 看似合理、實則錯得無聲，
  這是你研究裡**最大的隱形誤差來源**。

本 notebook 用 **合成 (synthetic) pseudo-Kinatrax** 資料，走完：
1. 生成 ground-truth OpenSim 角度，再**故意**重編碼成另一種慣例（模擬 Kinatrax 匯出）。
2. 展示「錯誤作法」：直接倒進同名 `.mot`，動作壞掉。
3. 用 `convention_mapping` 建 `ConventionMap` 把慣例映射回來，寫出正確 coordinates `.mot`。
4. **Round-trip 驗證**：回推角度應與 ground truth 幾乎相等。
5. 你的**真實 Kinatrax 檔案**該接在哪、以及慣例文件化 checklist。

> ⚙️ 步驟 1、3、4 的**核心對應與驗證邏輯不需 OpenSim**（純 numpy/scipy）；
> 寫檔與「動作壞掉」的視覺化 cell 需要 OpenSim 與一個上肢模型。


In [ ]:
# --- 讓 notebook 找得到本章的 src/ 模組 (put chapter src/ on sys.path) ---
import sys, pathlib
CHAPTER = pathlib.Path.cwd()
if CHAPTER.name == "notebooks":
    CHAPTER = CHAPTER.parent          # 允許從 notebooks/ 內啟動
sys.path.insert(0, str(CHAPTER / "src"))

import numpy as np
import matplotlib.pyplot as plt

import osim_kinematics_io as kio
import convention_mapping as cm
from convention_mapping import EulerTriplet, ConventionMap, euler_reorder, apply_map, round_trip_error
print("modules loaded")


## 1. 生成合成 pseudo-Kinatrax 角度流

先造一段平滑、投球風格的 **OpenSim ground-truth** 角度（3-DOF 肩 + 1-DOF 肘），
再套一組**我們假裝不知道**的慣例差異，反推出「Kinatrax 會輸出的」角度：

- 肩：目標 OpenSim Cardan 序列 `ZXY`（`plane_elv`, `shoulder_elv`, `axial_rot`，Holzbaur 家族命名），
  來源 = ISB humerothoracic proper-Euler `YXY`。
- 每軸再加 sign flip 與 zero-pose offset，全部以「度」表示。

反推方式：因 forward map 為 $q_{\text{target}}=\text{sign}\cdot\text{reorder}(q_{\text{src}})+\text{offset}$，
故 $q_{\text{src}}=\text{reorder}^{-1}\!\big((q_{\text{target}}-\text{offset})/\text{sign}\big)$。


In [ ]:
fs = 200.0
t = np.arange(0.0, 1.0, 1.0 / fs)
bump = (1 - np.cos(2 * np.pi * 0.5 * t)) / 2.0
truth = {                                   # ground-truth OpenSim coords (deg)
    "plane_elv":    30 + 20 * bump,
    "shoulder_elv": 20 + 40 * bump,         # 中間軸 (ZXY 的 X)，維持在 [-90,90]
    "axial_rot":   -15 + 25 * bump,
    "r_elbow_flex": 10 + 90 * bump,
}
# 我們假裝不知道的真實慣例差異
SIGN   = {"plane_elv": -1., "shoulder_elv": +1., "axial_rot": +1., "r_elbow_flex": -1.}
OFFSET = {"plane_elv": 0.,  "shoulder_elv": 0.,  "axial_rot": 30., "r_elbow_flex": 90.}
SEQ_TO, SEQ_FROM = "ZXY", "YXY"             # OpenSim Cardan  <-  ISB humerothoracic

pre = np.column_stack([(truth[k] - OFFSET[k]) / SIGN[k]
                       for k in ("plane_elv", "shoulder_elv", "axial_rot")])
src_sh = np.atleast_2d(euler_reorder(pre, SEQ_TO, SEQ_FROM))   # 反轉序列
kinatrax = {                                # 「Kinatrax 匯出」的角度流 (deg)
    "KinaShoulderPlane": src_sh[:, 0],
    "KinaShoulderElev":  src_sh[:, 1],
    "KinaShoulderAxial": src_sh[:, 2],
    "KinaElbowFlex":    (truth["r_elbow_flex"] - OFFSET["r_elbow_flex"]) / SIGN["r_elbow_flex"],
}
print("pseudo-Kinatrax channels:", list(kinatrax))


## 2. 錯誤作法：把 Kinatrax 角度「直接倒進同名 coordinate」

假設有人（不小心）把 Kinatrax 的肘角當成 `r_elbow_flex`、肩三軸當成
`plane_elv/shoulder_elv/axial_rot` 直接寫檔。因為慣例不同，回推的動作會偏離
ground truth——這正是無聲的錯誤。下面用 `round_trip_error` 量化「naive 對應」與真值的差。


In [ ]:
naive = {                        # 直接改名、不做任何序列/sign/offset 修正
    "plane_elv":    kinatrax["KinaShoulderPlane"],
    "shoulder_elv": kinatrax["KinaShoulderElev"],
    "axial_rot":    kinatrax["KinaShoulderAxial"],
    "r_elbow_flex": kinatrax["KinaElbowFlex"],
}
err_naive = round_trip_error(truth, naive)
print("Naive (直接改名) vs truth，每軸誤差 (deg):")
for k in ("plane_elv", "shoulder_elv", "axial_rot", "r_elbow_flex", "overall"):
    print(f"  {k:14s} max={err_naive[k]['max']:8.2f}  rms={err_naive[k]['rms']:8.2f}")


## 3. 正確作法：建立 `ConventionMap` 並套用 SOP

把慣例差異拆成四個可獨立檢查的成分：`name_map`（1-DOF 改名）、per-axis `sign`、
per-axis `offset_deg`、以及每個 3-DOF 關節的 `EulerTriplet`（序列轉換）。這對應
`notes.md` Section C 的映射 SOP。


In [ ]:
shoulder_tri = EulerTriplet(
    source_coords=("KinaShoulderPlane", "KinaShoulderElev", "KinaShoulderAxial"),
    target_coords=("plane_elv", "shoulder_elv", "axial_rot"),
    seq_from=SEQ_FROM, seq_to=SEQ_TO,
)
cmap = ConventionMap(
    name_map={"KinaElbowFlex": "r_elbow_flex"},
    sign=dict(SIGN), offset_deg=dict(OFFSET),
    euler_triplets=[shoulder_tri],
)
mapped = apply_map(kinatrax, cmap)          # -> OpenSim-coordinate 角度 (deg)
print("mapped coords:", list(mapped))


In [ ]:
# 寫成 coordinates .mot (值為度 -> inDegrees=yes)。這份檔就是餵給 ID / MocoTrack 的輸入。
mot_path = str(CHAPTER / "data" / "pitch_mapped_coordinates.mot")
kio_ok = True
try:
    kio.coordinates_to_mot(t, mapped, mot_path, in_degrees=True)
    print("wrote", mot_path, " inDegrees:", kio.mot_is_in_degrees(mot_path))
except ImportError as e:
    kio_ok = False
    print("OpenSim 未安裝，略過寫檔 (對應/驗證邏輯不受影響):", e)


## 4. Round-trip 驗證

正確映射後，回推角度應與 ground truth 幾乎相等（~1e-10 deg，浮點誤差級）。
下圖把 naive（錯）與 mapped（對）疊在真值上對比。


In [ ]:
err_mapped = round_trip_error(truth, mapped)
print("Mapped (正確映射) vs truth，每軸誤差 (deg):")
for k in ("plane_elv", "shoulder_elv", "axial_rot", "r_elbow_flex", "overall"):
    print(f"  {k:14s} max={err_mapped[k]['max']:.2e}  rms={err_mapped[k]['rms']:.2e}")
assert err_mapped["overall"]["max"] < 1e-6, "round-trip 應接近零"

fig, ax = plt.subplots(1, 4, figsize=(14, 3), sharex=True)
for a, k in zip(ax, ("plane_elv", "shoulder_elv", "axial_rot", "r_elbow_flex")):
    a.plot(t, truth[k], "k-", lw=3, alpha=.35, label="truth")
    a.plot(t, naive[k], "r:", label="naive (wrong)")
    a.plot(t, mapped[k], "g--", label="mapped (correct)")
    a.set_title(k); a.set_xlabel("s")
ax[0].set_ylabel("deg"); ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()


## 5. 你的**真實** Kinatrax 檔案接在哪 + 慣例文件化 checklist

把上面的合成 `kinatrax` dict 換成讀你真實 Kinatrax 匯出的角度即可。但在信任映射前，
**必須**先把來源慣例逆向工程並用 round-trip 在**全 ROM**（尤其奇異點附近）驗證。

**慣例文件化 checklist（放進 `data/` 與檔案並存）**
- [ ] 單位：度或弧度？角速度單位？
- [ ] 每個關節的 segment 定義（近端/遠端座標系原點與軸向）。
- [ ] 每個 3-DOF 關節的 Euler/Cardan **順序**與 intrinsic/extrinsic。
- [ ] zero / reference pose（T-pose？解剖姿勢？）。
- [ ] 每軸正負向定義（哪個方向為正）。
- [ ] 已知校正姿勢（至少 2 個、避開奇異點）用來 `infer_convention` 反推 sign/offset。
- [ ] 全 ROM round-trip：`round_trip_error` 的 overall max 是否夠小？


In [ ]:
# 真實資料骨架 (示意)：讀 Kinatrax 匯出 -> 套同一個 cmap -> 寫 .mot
# import pandas as pd
# raw = pd.read_csv("data/subject01_pitch.csv")          # 你的 Kinatrax 匯出
# kinatrax_real = {ch: raw[ch].to_numpy() for ch in cmap.triplet_source_names() | set(cmap.name_map)}
# mapped_real = apply_map(kinatrax_real, cmap)
# kio.coordinates_to_mot(raw["time"].to_numpy(), mapped_real,
#                        "data/subject01_pitch_coordinates.mot", in_degrees=True)
print("見上方註解：把合成 dict 換成真實 Kinatrax 讀檔即可。")


## 小結
- Kinatrax 是**角度驅動**：不跑 IK，直接寫 coordinates `.mot`；但**慣例映射**是成敗關鍵。
- `ConventionMap` = `name_map` + `sign` + `offset_deg` + `EulerTriplet`(序列)。
- 一律用**全 ROM round-trip** 驗證，尤其 Y-X-Y 在 elevation 0/180° 的 gimbal lock 附近。
- 這份 `.mot` 是 Stage 4/5 Inverse Dynamics 與 Stage 8 MocoTrack 的輸入。
- 下一步：肩胛骨看不到，得**生成**——見 [05c](05c_scapula_kinematics_generation.ipynb)。
